Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Random Variables

> ⚠️ **Draft — pending instructor review.** The visuals below execute, but execution cannot verify proofs. An SPS instructor should vet the arguments in this notebook before it is taught. Remove this banner after review.

Probability is not a new subject — it is [Measure Theory](./Measure_Theory.ipynb) with total mass 1. This workshop makes that identification precise: random variables are measurable functions, expectation is the Lebesgue integral, and every distribution you've ever met is a pushforward measure. The reward: one clean framework under the noise models of [Kalman filtering](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) and the losses of [machine learning](../../Intro_Mach_Learn/README.md).

### Visual setup & helpers
*(Safe to re-run anytime.)*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Probability Spaces & Random Variables* (~35 min)
**Goal:** define probability as a measure; define random variables as measurable functions and distributions as pushforwards.
**Builds on:** [Measure Theory](./Measure_Theory.ipynb). &nbsp; **Feeds into:** Session 2 (distributions, expectation, moments).

---

## Probability Spaces

**Definition.** A *probability space* is a measure space $(\Omega, \mathcal{F}, P)$ with $P(\Omega) = 1$:

- $\Omega$ — the *sample space*: every way the experiment could turn out;
- $\mathcal{F}$ — a σ-algebra of *events* (the askable questions);
- $P$ — a measure assigning each event its probability.

Everything from [Measure Theory](./Measure_Theory.ipynb) transfers instantly: monotonicity, countable (sub)additivity, continuity from below/above (no finiteness caveat needed — everything is $\le 1$).

💡 **Intuition.** Why must $\mathcal{F}$ be a σ-algebra and not "all subsets"? Two reasons. Philosophically, $\mathcal{F}$ encodes *what is knowable* — you can ask about complements ("not A") and countable unions ("eventually some $A_n$"), which is exactly closure under σ-operations. Technically, for continuous experiments (Ω = [0,1]) Vitali's set already showed that measuring everything is impossible.

## Random Variables

💡 **Intuition.** A random variable is **not random and not a variable** — it's a fixed, deterministic *function* $X: \Omega \to \mathbb{R}$ that reports a number about the outcome. All the randomness lives in which $\omega \in \Omega$ the world serves up. "Measurable" is the fine print making the function compatible with the askable questions: events like $\{X \le x\}$ must belong to $\mathcal{F}$, or their probability is undefined.

**Definition.** $X: \Omega \to \mathbb{R}$ is a *random variable* if it is $\mathcal{F}$-measurable: $X^{-1}(B) \in \mathcal{F}$ for every Borel set $B \in \mathcal{B}(\mathbb{R})$. (Checking $\{X \le x\} \in \mathcal{F}$ for all $x$ suffices, since the half-lines generate $\mathcal{B}$.)

**Definition (distribution).** The *law* of $X$ is the pushforward measure on $\mathbb{R}$:

$$P_X(B) = P(X^{-1}(B)) = P(X \in B).$$

$P_X$ is a probability measure on $(\mathbb{R}, \mathcal{B})$ — check the axioms via preimages. This is the licence to forget $\Omega$: all computations about $X$ alone happen on the real line with $P_X$.

### Proof: The CDF characterizes the law

Define $F(x) = P(X \le x)$. Then $F$ is (1) non-decreasing, (2) right-continuous, and (3) has limits $0$ at $-\infty$ and $1$ at $+\infty$.

(1) $x \le y \implies \{X \le x\} \subseteq \{X \le y\}$; apply monotonicity.

(2) Let $x_n \downarrow x$. The events $\{X \le x_n\}$ decrease to $\bigcap_n \{X \le x_n\} = \{X \le x\}$; continuity from above gives $F(x_n) \to F(x)$.

(3) $\{X \le -n\} \downarrow \emptyset$ and $\{X \le n\} \uparrow \Omega$; apply continuity again. $\blacksquare$

*(Conversely, any $F$ with these properties is the CDF of some random variable — and since intervals generate $\mathcal{B}$, the CDF pins down the whole law. Note left-continuity can fail: $F$ jumps exactly at atoms, by size $P(X = x)$.)*

In [2]:
# One function, three personalities: CDFs of discrete, continuous, and mixed laws
x = np.linspace(-0.5, 3.5, 2000)

F_disc = np.select([x < 0, x < 1, x < 2], [0, 0.3, 0.8], default=1.0)       # atoms at 0,1,2
F_cont = np.clip(x / 3, 0, 1)                                                # Uniform[0,3]
F_mix  = 0.5 * F_disc + 0.5 * F_cont                                         # half-and-half

fig, axes = plt.subplots(1, 3, figsize=(10, 2.6), sharey=True)
for ax, F, t in zip(axes, [F_disc, F_cont, F_mix], ["discrete (jumps only)", "continuous", "mixed"]):
    ax.plot(x, F); ax.set_title(t); ax.grid(True)
axes[0].set_ylabel("F(x)")
plt.suptitle("Right-continuous, non-decreasing, 0→1 — that's ALL a CDF must be", y=1.05)
plt.tight_layout(); plt.show()

/tmp/ipykernel_1898601/641952581.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *Distributions, Expectation & Moments* (~40 min)
**Goal:** compute with densities; define expectation as a Lebesgue integral; prove Markov & Chebyshev.
**Builds on:** Session 1. &nbsp; **Feeds into:** [Independence](./README.md#6-independence-draft--pending-review).

---

## Densities

**Definition.** $X$ has *density* $f$ if $P(X \in B) = \int_B f \, d\lambda$ for all Borel $B$ — the law is "Lebesgue measure reweighted by $f$." Then $F' = f$ wherever $f$ is continuous. Not every law has one (atoms don't; stranger things — laws supported on the Cantor set — don't either), which is why the measure-theoretic definition, not the density, is primary.

## Expectation

💡 **Intuition.** Expectation is *literally* the Lebesgue integral of the previous notebook: $E[X] = \int_\Omega X \, dP$ — the y-axis-slicing integral, built from simple functions up. For a coin-flip payout it reduces to the weighted sum you learned as a kid; the machinery exists so that the *same* definition covers payouts with densities, atoms, or neither — and so MCT/DCT let us swap limits and expectations.

**Change of variables (LOTUS).** For measurable $g$ with $g(X) \in L^1$:

$$E[g(X)] = \int_\Omega g(X(\omega)) \, dP(\omega) = \int_{\mathbb{R}} g(x) \, dP_X(x) \;\; \Big( = \int g(x) f(x) \, dx \text{ if a density exists} \Big)$$

*Proof route:* true for $g = \mathbf{1}_B$ by definition of pushforward; extend to simple $g$ by linearity, to $g \ge 0$ by MCT, to general $g$ by splitting $g^{\pm}$. This "indicator → simple → limit" ladder is *the* standard proof pattern of measure theory — you'll use it constantly.

**Moments.** $E[X^k]$ is the $k$-th moment; variance is $\mathrm{Var}(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$ (expand the square, use linearity).

### Proof: Markov & Chebyshev inequalities

**Markov.** For $X \ge 0$ and $a > 0$: $\;P(X \ge a) \le \frac{E[X]}{a}$.

Pointwise, $a \, \mathbf{1}_{\{X \ge a\}} \le X$ (if $X(\omega) \ge a$ the left side is $a$; otherwise it's $0$). Take expectations — monotonicity of the integral gives $a \, P(X \ge a) \le E[X]$. $\blacksquare$

**Chebyshev.** Apply Markov to the non-negative variable $(X - E[X])^2$ with threshold $a^2$:

$$P(|X - E[X]| \ge a) = P\big((X - E[X])^2 \ge a^2\big) \le \frac{\mathrm{Var}(X)}{a^2}. \;\blacksquare$$

One line each — and Chebyshev is the engine of the weak law of large numbers in [Independence](./README.md#6-independence-draft--pending-review).

In [3]:
# How tight is Chebyshev? Empirical tail vs the bound, for a standard normal
a_grid = np.linspace(0.5, 4, 30)
samples = rng.standard_normal(200_000)
empirical = [(np.abs(samples) >= a).mean() for a in a_grid]

plt.figure(figsize=(7.5, 3))
plt.semilogy(a_grid, empirical, label="empirical  P(|X| ≥ a),  X ~ N(0,1)")
plt.semilogy(a_grid, np.minimum(1, 1 / a_grid**2), "--", label="Chebyshev bound  1/a²")
plt.legend(); plt.grid(True); plt.xlabel("a")
plt.title("Chebyshev is loose for Gaussians — its power is needing ONLY a variance")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1898601/1205581157.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Sampling meets the theory

A density is not a histogram — but histograms of samples converge to the density, and empirical means converge to $E[X]$ (that's the law of large numbers, proven properly next workshop).

In [4]:
# Exponential(1): compare sample histogram/mean/variance to the theory
X = rng.exponential(1.0, size=100_000)
x = np.linspace(0, 6, 300)

plt.figure(figsize=(7.5, 2.8))
plt.hist(X, bins=80, range=(0, 6), density=True, alpha=0.5, label="100k samples")
plt.plot(x, np.exp(-x), "k", label="density $e^{-x}$")
plt.legend(); plt.title("Exponential(1)")
plt.tight_layout(); plt.show()

print(f"E[X]:   theory 1.000   sample {X.mean():.3f}")
print(f"Var(X): theory 1.000   sample {X.var():.3f}")
print(f"P(X≥3): theory {np.exp(-3):.4f}  sample {(X >= 3).mean():.4f}  Markov bound {1/3:.3f}")

E[X]:   theory 1.000   sample 1.004
Var(X): theory 1.000   sample 1.025
P(X≥3): theory 0.0498  sample 0.0511  Markov bound 0.333


/tmp/ipykernel_1898601/2728721527.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## Where next

- [Independence](./README.md#6-independence-draft--pending-review) — product measures, Borel–Cantelli, and the laws of large numbers that justify every Monte Carlo estimate above.
- [Adaptive Filtering: Kalman](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) — the $Q$, $R$ matrices are covariances of exactly these objects.
- [Measure Theory](./Measure_Theory.ipynb) — the machinery, if you skipped ahead.